In [2]:
API_key="API_KEY"

In [3]:
import os
import json
import re
from google import genai
from google.genai import types

# Setup Gemini client and model
client = genai.Client(api_key=API_key)
MODEL_NAME = "models/gemini-2.0-flash"
print(f"Using model: {MODEL_NAME} for API generation")

# Instruction template
INSTRUCTIONS = (
    "Je krijgt een JSON-array met documenten met betrekking tot de coronacrisis in het Nederlands of Engels.\n"
    "Geef de resultaten terug in het Nederlands. Voor elk document extraheer je de volgende informatie:\n"
    "1. bevolkingsgroepen: alle verwijzingen naar specifieke groepen mensen die in het document genoemd worden en relevant zijn voor de coronacrisis of de besluitvorming eromheen. Dit kunnen groepen zijn op basis van leeftijd (bijv. 'ouderen', 'jongeren', 'kinderen'), gezondheidstoestand (bijv. 'patiënten', 'besmette personen', 'kwetsbaren'), beroep (bijv. 'zorgpersoneel', 'onderwijzers'), of andere relevante kenmerken (bijv. 'mensen met een migratieachtergrond', 'daklozen').\n"
    "**BELANGRIJK: Identificeer groepen mensen op basis van de context van het document. Let op de manier waarop de groepen worden gedefinieerd of beschreven in de tekst. Het is belangrijk om de nuances van de taal te begrijpen en te interpreteren welke groepen mensen relevant zijn in de context van de coronacrisis.**\n"
    "**EXTRACTIE:** Zoek naar verwijzingen naar bevolkingsgroepen die LETTERLIJK in het document voorkomen. De verwijzingen moeten minimaal 2 karakters lang zijn.\n"
    "**NORMALISATIE (INDIEN NODIG):** Soms worden bevolkingsgroepen op verschillende manieren aangeduid in een document. Als er verschillende termen gebruikt worden om naar dezelfde groep mensen te verwijzen, kies dan de meest gangbare of expliciete term. Bijvoorbeeld, als een document zowel 'senioren' als 'ouderen' noemt, gebruik dan 'ouderen' (tenzij 'senioren' de meer gebruikelijke term is in die context). Normalisatie is echter niet altijd nodig; als de termen duidelijk verschillend zijn, bewaar ze dan afzonderlijk.\n"
    "**VOORKOM OVERLAP: Probeer overlappende of redundante groepen te vermijden. Als 'kwetsbare ouderen' genoemd wordt, overweeg dan of dit hetzelfde is als 'ouderen' en zo ja, gebruik dan alleen 'ouderen'. Als er een duidelijk onderscheid is, neem dan beide groepen op.**\n"
    "**EXTREEM BELANGRIJK: Genereer ABSOLUUT GEEN bevolkingsgroepen die NIET in het document voorkomen. De normalisatie mag NOOIT leiden tot het toevoegen van groepen die niet al op de een of andere manier in het document genoemd worden. Indien er geen relevante bevolkingsgroepen in het document genoemd worden, laat de \"bevolkingsgroepen\" array dan leeg.**\n"
    "Stuur als output één JSON-object met een \"results\"-array. De \"results\"-array bevat objecten met een \"document_id\" en een \"bevolkingsgroepen\" array. De \"bevolkingsgroepen\" array bevat een lijst van strings, waarbij elke string een (genormaliseerde) aanduiding van een bevolkingsgroep is.\n"
    "Neem per document het document_id mee in de response.\n"
    "Lever uitsluitend geldige JSON zonder extra markdown‑fences, zonder trailing commas, met alle strings correct geescaped.\n"
    "Hier zijn een paar voorbeelden:\n"
    "INPUT DOCUMENT 1: 'De ouderen zijn extra kwetsbaar voor het virus.'\n"
    "INPUT DOCUMENT 2: 'Kinderen en jongeren hebben minder vaak ernstige symptomen.'\n"
    "INPUT DOCUMENT 3: 'Het zorgpersoneel werkt hard om de patiënten te helpen.'\n"
    "INPUT DOCUMENT 4: 'Kwetsbare ouderen hebben extra bescherming nodig.'\n"
    "INPUT DOCUMENT 5: 'Dit document bespreekt de impact van corona op de economie.'\n"
    "OUTPUT 1: {\"results\": [{\"document_id\": \"voorbeeld-1\", \"bevolkingsgroepen\": [\"ouderen\"]}]}\n"
    "OUTPUT 2: {\"results\": [{\"document_id\": \"voorbeeld-2\", \"bevolkingsgroepen\": [\"kinderen\", \"jongeren\"]}]}\n"
    "OUTPUT 3: {\"results\": [{\"document_id\": \"voorbeeld-3\", \"bevolkingsgroepen\": [\"zorgpersoneel\", \"patiënten\"]}]}\n"
    "OUTPUT 4: {\"results\": [{\"document_id\": \"voorbeeld-4\", \"bevolkingsgroepen\": [\"kwetsbare ouderen\"]}]}\n"
    "OUTPUT 5: {\"results\": [{\"document_id\": \"voorbeeld-5\", \"bevolkingsgroepen\": []}]}\n"
)

# Batch directory and range
BATCH_DIR = "/home/nena-meijer/PyCharmMiscProject/information_extraction/batches"
BATCH_START = 2501
BATCH_END = 3813  # inclusive

aggregated_results = []

for i in range(BATCH_START, BATCH_END + 1):
    batch_path = os.path.join(BATCH_DIR, f"batch_{i}.json")
    if not os.path.isfile(batch_path):
        print(f"Batch {i}: bestand niet gevonden, overslaan...")
        continue

    with open(batch_path, 'r', encoding='utf-8') as f:
        try:
            batch = json.load(f)
        except json.JSONDecodeError as e:
            print(f"Batch {i}: JSON decode error bij het inlezen van bestand: {e}")
            aggregated_results.append({'batch': i, 'error': f'File load error: {e}', 'raw_response': None})
            continue

    prompt = INSTRUCTIONS + json.dumps({'documents': batch}, ensure_ascii=False, indent=2)
    print(len(prompt))
    print(f"Processing batch {i} with {len(batch)} documents...")

    contents = [types.Content(role="user", parts=[types.Part.from_text(text=prompt)])]
    config = types.GenerateContentConfig(response_mime_type="text/plain")

    try:
        response_text = ''.join(
            chunk.text for chunk in client.models.generate_content_stream(
                model=MODEL_NAME, contents=contents, config=config
            )
        )

        # Clean up potential Markdown code fences
        cleaned = response_text.strip()
        cleaned = re.sub(r"^```json", "", cleaned, flags=re.MULTILINE)
        cleaned = re.sub(r"```$", "", cleaned, flags=re.MULTILINE)
        cleaned = cleaned.strip()

        data = json.loads(cleaned)
        results = data.get('results', [])
        print(f"Batch {i}: parsed {len(results)} results")
        aggregated_results.extend(results)

    except json.JSONDecodeError as e:
        print(f"Batch {i} JSON parse error: {e}\nIncluding raw response in results.json")
        aggregated_results.append({'batch': i, 'error': str(e), 'raw_response': cleaned})

    except Exception as e:
        print(f"Batch {i}: onverwachte fout: {e}")
        aggregated_results.append({'batch': i, 'error': str(e), 'raw_response': None})

# Save final aggregated results
output_path = '/home/nena-meijer/PyCharmMiscProject/information_extraction/groups/results_groups_2501_tm_3813.json'
with open(output_path, 'w', encoding='utf-8') as outfile:
    json.dump({'results': aggregated_results}, outfile, ensure_ascii=False, indent=2)

print(f"Saved aggregated results: {len(aggregated_results)} entries to '{output_path}'")


Using model: models/gemini-2.0-flash for API generation
46543
Processing batch 2501 with 1 documents...
Batch 2501: parsed 6 results
33279
Processing batch 2502 with 1 documents...
Batch 2502: parsed 6 results
23598
Processing batch 2503 with 1 documents...
Batch 2503: parsed 6 results
34890
Processing batch 2504 with 1 documents...
Batch 2504: parsed 6 results
27106
Processing batch 2505 with 1 documents...
Batch 2505: parsed 6 results
28685
Processing batch 2506 with 1 documents...
Batch 2506: parsed 6 results
59947
Processing batch 2507 with 1 documents...
Batch 2507: parsed 6 results
75167
Processing batch 2508 with 1 documents...
Batch 2508: parsed 6 results
28320
Processing batch 2509 with 1 documents...
Batch 2509: parsed 6 results
75696
Processing batch 2510 with 1 documents...
Batch 2510: parsed 6 results
37006
Processing batch 2511 with 1 documents...
Batch 2511: parsed 6 results
35262
Processing batch 2512 with 1 documents...
Batch 2512: parsed 6 results
47190
Processing bat

In [1]:
import json
import glob

# Lijst van al je JSON-bestanden, bijvoorbeeld alle .json in een map
json_files = glob.glob('/home/nena-meijer/PyCharmMiscProject/information_extraction/groups/*.json')

# Gecombineerde resultaten
combined_results = []

for file in json_files:
    with open(file, 'r', encoding='utf-8') as f:
        data = json.load(f)
        combined_results.extend(data['results'])

# Totale lengte van combined_results
print("Totaal aantal items in results:", len(combined_results))

# Optioneel: schrijf naar één bestand
with open('groups_all.json', 'w', encoding='utf-8') as f_out:
    json.dump({'results': combined_results}, f_out, ensure_ascii=False, indent=2)


Totaal aantal items in results: 22869


In [18]:
import json
import csv
import re

# Paden
json_path = '/home/nena-meijer/PyCharmMiscProject/information_extraction/groups/groups_all.json'
csv_path = '/home/nena-meijer/PyCharmMiscProject/database/Group_per_doc.csv'

# JSON inladen
with open(json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

valid_results = []
extra_results = []
failed_items = []

def try_fix_json(raw, idx):
    raw_fixed = raw.replace("\\'", "'")
    raw_fixed = re.sub(r',\s*}', '}', raw_fixed)
    raw_fixed = re.sub(r'}\s*{', '},{', raw_fixed)

    # 🔥 Specifieke harde fix voor item 13682
    if idx == 13682:
        print(f"Item {idx}: Forceer harde afsluiting van afgebroken JSON.")
        raw_fixed = re.sub(r'"zorg[^"]*$', '"zorg"]}', raw_fixed)
        if '"results": [' not in raw_fixed:
            raw_fixed = '{"results": [' + raw_fixed
        # Vind de laatste goede afsluiting
        if raw_fixed.count('}') > raw_fixed.count(']}'):
            raw_fixed = raw_fixed + ']}'
        raw_fixed = raw_fixed.split('}]}')[0] + '}]}'
        print(raw_fixed)

    # Normale afsluit fixes
    if '"zorg' in raw_fixed and not raw_fixed.endswith('"]}'):
        raw_fixed = re.sub(r'"zorg[^"]*$', '"zorg"]}', raw_fixed)

    if not raw_fixed.strip().endswith(']}'):
        raw_fixed = raw_fixed.rstrip(', \n') + ']}'

    if '"results"' not in raw_fixed:
        raw_fixed = '{"results": [' + raw_fixed + ']}'

    try:
        parsed_raw = json.loads(raw_fixed)
        return parsed_raw['results']
    except json.JSONDecodeError as e:
        print(f"Item {idx}: JSONDecodeError: {e}")
        try:
            from json import JSONDecoder
            decoder = JSONDecoder()
            obj, idx_end = decoder.raw_decode(raw_fixed)
            print(f"Item {idx}: Alleen eerste JSON object gelezen.")
            return obj['results']
        except Exception as e2:
            print(f"Item {idx}: Kon niet repareren: {e2}")
            return None


# Verwerk data
for idx, item in enumerate(data['results']):
    if 'document_id' in item:
        valid_results.append(item)
    elif 'raw_response' in item and item['raw_response']:
        raw = item['raw_response']
        fixed_results = try_fix_json(raw, idx)
        if fixed_results:
            extra_results.extend(fixed_results)
        else:
            failed_items.append({'index': idx, 'error': 'Could not parse'})

# Combineer resultaten
all_results = valid_results + extra_results

# Schrijf CSV
with open(csv_path, 'w', encoding='utf-8', newline='') as f_csv:
    writer = csv.writer(f_csv)
    writer.writerow(['document_id', 'name'])

    for item in all_results:
        document_id = item['document_id']
        groepen = item.get('bevolkingsgroepen', [])
        if groepen:
            for groep in groepen:
                writer.writerow([document_id, groep])
        else:
            writer.writerow([document_id, ''])

print(f"Aantal geldige items: {len(all_results)}")
print(f"Niet geparste raw_responses: {len(failed_items)}")
print(f"CSV opgeslagen als: {csv_path}")


Item 6714: JSONDecodeError: Extra data: line 40 column 2 (char 650)
Item 6714: Alleen eerste JSON object gelezen.
Item 13682: Forceer harde afsluiting van afgebroken JSON.
{
  "results": [
    {
      "document_id": "187-3",
      "bevolkingsgroepen": []
    },
    {
      "document_id": "187-4",
      "bevolkingsgroepen": [
        "zwangere",
        "minderjarige",
        "huisartsen",
        "huisgenoten",
        "zorgmedewerkers",
        "personen ouder dan 70 jaar",
        "volwassenen",
        "arbeidsmigranten",
        "kinderen",
        "jongeren",
        "indexen",
        "contacten",
        "patient"
      ]
    },
    {
      "document_id": "187-5",
      "bevolkingsgroepen": [
        "ouderen",
        "patienten",
        "volwassenen",
        "mensen"
      ]
    },
    {
      "document_id": "187-6",
      "bevolkingsgroepen": [
        "personen boyen de 18 jaar",
        "personen onder de 18 jaar",
        "ouder dan 65 jaar",
        "artsen",
        "

In [19]:
import csv
from collections import Counter

csv_path = '/home/nena-meijer/PyCharmMiscProject/database/Group_per_doc.csv'

namen = []

# Lees de CSV en verzamel namen
with open(csv_path, 'r', encoding='utf-8') as f_csv:
    reader = csv.DictReader(f_csv)
    for row in reader:
        name = row['name'].strip()
        if name:  # alleen niet-lege namen
            namen.append(name)

# Tel de namen
naam_tellingen = Counter(namen)

# Sorteer op count (aflopend)
gesorteerd = naam_tellingen.most_common()

# Print de resultaten
print("Unieke namen en hun aantallen (gesorteerd):")
for naam, count in gesorteerd:
    print(f"{naam}: {count}")

print(f"Totaal unieke namen: {len(naam_tellingen)}")


Unieke namen en hun aantallen (gesorteerd):
mensen: 1636
kinderen: 1501
patienten: 1471
patiënten: 1241
medewerkers: 1124
ouderen: 1084
zorgmedewerkers: 840
jongeren: 821
personeel: 623
burgers: 601
personen: 582
patient: 515
clienten: 503
zorgverleners: 502
ouders: 439
mantelzorgers: 415
huisartsen: 409
jeugd: 401
zorgpersoneel: 386
reizigers: 365
bewoners: 364
werknemers: 323
zorgprofessionals: 309
nederlanders: 292
kwetsbare groepen: 286
artsen: 281
toeristen: 275
volwassenen: 269
mensen met een beperking: 257
verpleegkundigen: 247
arbeidsmigranten: 245
inwoners: 239
collega's: 233
kwetsbare personen: 230
huisgenoten: 227
studenten: 222
kwetsbare mensen: 219
zorgaanbieders: 219
kwetsbare ouderen: 208
passagiers: 207
bevolking: 203
professionals: 191
risicogroepen: 179
bezoekers: 173
cliënten: 166
naasten: 165
coronapatienten: 164
leerlingen: 157
gezinnen: 154
vrijwilligers: 132
client: 128
burger: 126
ondernemers: 123
familie: 114
deelnemers: 113
ziekenhuizen: 113
kwetsbaren: 111
we

In [20]:
import json
import csv
from collections import Counter

# Pad naar je mapping JSON
mapping_path = '/home/nena-meijer/PyCharmMiscProject/information_extraction/groups/group_mapping.json'
# Pad naar je CSV
csv_path = '/home/nena-meijer/PyCharmMiscProject/database/Group_per_doc.csv'

# Mapping inladen
with open(mapping_path, 'r', encoding='utf-8') as f_map:
    naam_mapping = json.load(f_map)

# Mapping omzetten naar dictionary voor snelle lookup
lookup = {}
for group in naam_mapping:
    main_name = list(group.keys())[0]
    for variant in group[main_name]:
        lookup[variant.lower()] = main_name  # case-insensitive

namen_genormaliseerd = []

# Lees CSV en pas mapping toe
with open(csv_path, 'r', encoding='utf-8') as f_csv:
    reader = csv.DictReader(f_csv)
    for row in reader:
        name = row['name'].strip()
        if name:
            name_lower = name.lower()
            if name_lower in lookup:
                namen_genormaliseerd.append(lookup[name_lower])
            else:
                namen_genormaliseerd.append(name)  # niet in mapping, origineel laten

# Tel de genormaliseerde namen
naam_tellingen = Counter(namen_genormaliseerd)

# Gesorteerd afdrukken
gesorteerd = naam_tellingen.most_common()

print("Genormaliseerde namen en hun aantallen:")
for naam, count in gesorteerd:
    print(f"{naam}: {count}")

print(f"\nTotaal unieke genormaliseerde namen: {len(naam_tellingen)}")


Genormaliseerde namen en hun aantallen:
patiënten: 2712
mensen: 1636
kinderen: 1501
medewerkers: 1124
ouderen: 1084
zorgmedewerkers: 840
jongeren: 821
personeel: 623
burgers: 601
personen: 582
patient: 515
clienten: 503
zorgverleners: 502
ouders: 439
mantelzorgers: 415
huisartsen: 409
jeugd: 401
zorgpersoneel: 386
reizigers: 365
bewoners: 364
werknemers: 323
zorgprofessionals: 309
nederlanders: 292
kwetsbare groepen: 286
artsen: 281
toeristen: 275
volwassenen: 269
mensen met een beperking: 257
verpleegkundigen: 247
arbeidsmigranten: 245
inwoners: 239
collega's: 233
kwetsbare personen: 230
huisgenoten: 227
studenten: 222
kwetsbare mensen: 219
zorgaanbieders: 219
kwetsbare ouderen: 208
passagiers: 207
bevolking: 203
professionals: 191
risicogroepen: 179
bezoekers: 173
cliënten: 166
naasten: 165
coronapatienten: 164
leerlingen: 157
gezinnen: 154
vrijwilligers: 132
client: 128
burger: 126
ondernemers: 123
familie: 114
deelnemers: 113
ziekenhuizen: 113
kwetsbaren: 111
werkgevers: 107
vrouwe

In [22]:
import json
import csv
from collections import Counter

# Paden
mapping_path = '/home/nena-meijer/PyCharmMiscProject/information_extraction/groups/group_mapping.json'
csv_input_path = '/home/nena-meijer/PyCharmMiscProject/database/Group_per_doc.csv'
csv_output_path = '/home/nena-meijer/PyCharmMiscProject/database/Group.csv'

# Mapping inladen
with open(mapping_path, 'r', encoding='utf-8') as f_map:
    naam_mapping = json.load(f_map)

# Mapping omzetten naar dictionary
lookup = {}
for group in naam_mapping:
    main_name = list(group.keys())[0]
    for variant in group[main_name]:
        lookup[variant.lower()] = main_name

namen_genormaliseerd = []

# Lees CSV en pas mapping toe
with open(csv_input_path, 'r', encoding='utf-8') as f_csv:
    reader = csv.DictReader(f_csv)
    for row in reader:
        name = row['name'].strip()
        if name:
            name_lower = name.lower()
            if name_lower in lookup:
                namen_genormaliseerd.append(lookup[name_lower])
            else:
                namen_genormaliseerd.append(name)

# Tel en haal unieke namen
unieke_namen = sorted(set(namen_genormaliseerd))

# Schrijf unieke namen naar CSV
with open(csv_output_path, 'w', encoding='utf-8', newline='') as f_out:
    writer = csv.writer(f_out)
    writer.writerow(['group_id', 'name'])  # header
    for idx, naam in enumerate(unieke_namen, start=1):
        writer.writerow([idx, naam])

print(f"Unieke namen opgeslagen in: {csv_output_path}")
print(f"Totaal unieke namen: {len(unieke_namen)}")


Unieke namen opgeslagen in: /home/nena-meijer/PyCharmMiscProject/database/Group.csv
Totaal unieke namen: 4578


In [24]:
import json
import csv

# Paden naar je bestanden
mapping_path = '/home/nena-meijer/PyCharmMiscProject/information_extraction/groups/group_mapping.json'
csv_input_path = '/home/nena-meijer/PyCharmMiscProject/database/Group_per_doc.csv'
csv_output_path = '/home/nena-meijer/PyCharmMiscProject/database/Group_normalized.csv'

# Mapping inladen
with open(mapping_path, 'r', encoding='utf-8') as f_map:
    naam_mapping = json.load(f_map)

# Mapping omzetten naar dictionary voor snelle lookup
lookup = {}
for group in naam_mapping:
    main_name = list(group.keys())[0]
    for variant in group[main_name]:
        lookup[variant.lower()] = main_name  # case-insensitive mapping

# Nieuwe lijst voor rijen met genormaliseerde namen
genormaliseerde_rijen = []

# Lees CSV en pas mapping toe
with open(csv_input_path, 'r', encoding='utf-8') as f_csv:
    reader = csv.DictReader(f_csv)
    for row in reader:
        name = row['name'].strip()
        if name:
            name_lower = name.lower()
            if name_lower in lookup:
                row['name'] = lookup[name_lower]  # vervang met genormaliseerde naam
            else:
                row['name'] = name  # geen match, laat origineel staan
        else:
            row['name'] = ''  # lege waarde blijft leeg
        genormaliseerde_rijen.append(row)

# Schrijf het resultaat naar een nieuwe CSV
with open(csv_output_path, 'w', encoding='utf-8', newline='') as f_out:
    fieldnames = ['document_id', 'name']
    writer = csv.DictWriter(f_out, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(genormaliseerde_rijen)

print(f"Genormaliseerde CSV opgeslagen als: {csv_output_path}")


Genormaliseerde CSV opgeslagen als: /home/nena-meijer/PyCharmMiscProject/database/Group_normalized.csv


In [25]:
import csv

# Paden naar je CSV's
person_csv_path = '/home/nena-meijer/PyCharmMiscProject/database/Group_normalized.csv'
unique_persons_csv_path = '/home/nena-meijer/PyCharmMiscProject/database/Group.csv'
output_csv_path = '/home/nena-meijer/PyCharmMiscProject/database/DocumentGroup.csv'

# Stap 1: Laad person_id -> name mapping
person_mapping = {}
with open(unique_persons_csv_path, 'r', encoding='utf-8') as f_unique:
    reader = csv.DictReader(f_unique)
    for row in reader:
        person_mapping[row['name'].strip()] = row['group_id']

# Stap 2: Verwerk de document_id, name CSV
document_person_rows = []

with open(person_csv_path, 'r', encoding='utf-8') as f_persons:
    reader = csv.DictReader(f_persons)
    for row in reader:
        document_id = row['document_id']
        name = row['name'].strip()
        if name and name in person_mapping:
            person_id = person_mapping[name]
            document_person_rows.append({'document_id': document_id, 'group_id': person_id})
        elif name == '':
            # Optioneel: sla lege namen over, of voeg document_id met lege person_id toe
            continue
        else:
            # Naam niet gevonden in mapping, optioneel loggen
            print(f"Naam niet gevonden: {name}")

# Stap 3: Schrijf nieuwe CSV met document_id, person_id
with open(output_csv_path, 'w', encoding='utf-8', newline='') as f_out:
    writer = csv.DictWriter(f_out, fieldnames=['document_id', 'group_id'])
    writer.writeheader()
    writer.writerows(document_person_rows)

print(f"Document-Person CSV opgeslagen als: {output_csv_path}")
print(f"Totaal koppelingen: {len(document_person_rows)}")


Document-Person CSV opgeslagen als: /home/nena-meijer/PyCharmMiscProject/database/DocumentGroup.csv
Totaal koppelingen: 39614


In [26]:
import csv

# Pad naar je DocumentPerson.csv
document_person_csv_path = '/home/nena-meijer/PyCharmMiscProject/database/DocumentGroup.csv'

# Verzamel alle person_id's
person_ids = set()

with open(document_person_csv_path, 'r', encoding='utf-8') as f_csv:
    reader = csv.DictReader(f_csv)
    for row in reader:
        person_ids.add(row['group_id'])

# Resultaat
print(f"Totaal unieke person_id's in DocumentPerson: {len(person_ids)}")


Totaal unieke person_id's in DocumentPerson: 4578
